# 🛒 Customer Segmentation — E-Commerce Retail
### RFM Analysis + K-Means Clustering for Customer Intelligence

---

**Author:** Your Name  
**Dataset:** Online Retail II — UCI / Kaggle  
**Tools:** Python, pandas, scikit-learn, matplotlib, seaborn  
**Goal:** Segment customers using RFM (Recency, Frequency, Monetary) analysis and K-Means clustering to enable targeted marketing strategies

---

## 📌 Table of Contents
1. [Business Understanding](#1)
2. [Import Libraries](#2)
3. [Load & Explore Data](#3)
4. [Data Cleaning](#4)
5. [Exploratory Data Analysis (EDA)](#5)
6. [RFM Feature Engineering](#6)
7. [K-Means Clustering](#7)
8. [Cluster Analysis & Profiling](#8)
9. [Business Recommendations](#9)
10. [Conclusion](#10)

---
## 1. Business Understanding <a id='1'></a>

### Problem Statement
Not all customers are equal. A one-size-fits-all marketing approach wastes budget on low-value customers while under-investing in high-value ones. **Customer segmentation** solves this by grouping customers with similar behaviour into distinct segments.

### Objective
Use **RFM Analysis** (Recency, Frequency, Monetary) and **K-Means Clustering** to identify distinct customer segments from transaction data, and translate those segments into actionable marketing strategies.

### RFM Framework
| Dimension | Definition | Business Meaning |
|---|---|---|
| **Recency (R)** | Days since last purchase | How recently did the customer buy? |
| **Frequency (F)** | Total number of orders | How often do they buy? |
| **Monetary (M)** | Total amount spent | How much do they spend? |

### Business Value
| Segment | Strategy | Expected ROI |
|---|---|---|
| Champions | Reward & retain | Highest LTV |
| At Risk | Win-back campaigns | Prevent churn |
| New Customers | Onboarding nurture | Convert to loyal |
| Lost Customers | Re-engagement or drop | Cut wasted spend |

---
## 2. Import Libraries <a id='2'></a>

In [ ]:
# ── Core ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# ── Preprocessing & Clustering ────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# ── Date handling ─────────────────────────────────────
from datetime import datetime

print('✅ All libraries imported successfully')

---
## 3. Load & Explore Data <a id='3'></a>

In [ ]:
# ── Load dataset ──────────────────────────────────────
# The dataset comes as an Excel file with two sheets
# We load both years and combine them

try:
    # Try loading Excel directly
    df_09 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010')
    df_10 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011')
    df = pd.concat([df_09, df_10], ignore_index=True)
    print('✅ Loaded from Excel file')
except FileNotFoundError:
    try:
        # Try CSV fallback
        df = pd.read_csv('online_retail_II.csv')
        print('✅ Loaded from CSV file')
    except:
        print('❌ File not found. Make sure online_retail_II.xlsx is in the same folder as this notebook.')

print(f'\nDataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(10)

In [ ]:
# ── Column names & types ──────────────────────────────
print('📋 Column Names & Data Types')
print('─' * 45)
print(df.dtypes)
print(f'\nDate Range: {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
print(f'Unique Customers: {df["Customer ID"].nunique():,}')
print(f'Unique Products:  {df["StockCode"].nunique():,}')
print(f'Unique Countries: {df["Country"].nunique()}')

In [ ]:
# ── Statistical Summary ───────────────────────────────
print('📊 Statistical Summary')
df.describe()

In [ ]:
# ── Missing Values ────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('❗ Missing Values:')
print(missing_df.to_string())

---
## 4. Data Cleaning <a id='4'></a>

In [ ]:
print(f'Shape before cleaning: {df.shape}')

# ── Step 1: Drop rows with missing Customer ID ────────
# We cannot compute RFM without a customer identifier
df = df.dropna(subset=['Customer ID'])
print(f'After dropping missing Customer ID: {df.shape}')

# ── Step 2: Remove cancelled invoices ─────────────────
# Invoices starting with 'C' are cancellations
df = df[~df['Invoice'].astype(str).str.startswith('C')]
print(f'After removing cancellations: {df.shape}')

# ── Step 3: Remove negative / zero quantities ─────────
df = df[df['Quantity'] > 0]
print(f'After removing invalid quantities: {df.shape}')

# ── Step 4: Remove negative / zero prices ─────────────
df = df[df['Price'] > 0]
print(f'After removing invalid prices: {df.shape}')

# ── Step 5: Remove test / non-product stock codes ─────
invalid_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT']
df = df[~df['StockCode'].astype(str).str.upper().isin(invalid_codes)]
print(f'After removing invalid stock codes: {df.shape}')

# ── Step 6: Ensure correct data types ─────────────────
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)

# ── Step 7: Create TotalPrice column ──────────────────
df['TotalPrice'] = df['Quantity'] * df['Price']

print(f'\n✅ Final clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# ── Focus on UK customers (largest market) ────────────
print(f'Top 10 Countries by transaction count:')
print(df['Country'].value_counts().head(10).to_string())

print(f'\nUK transactions: {(df["Country"] == "United Kingdom").sum():,}')
print(f'UK % of total  : {(df["Country"] == "United Kingdom").mean()*100:.1f}%')

# Keep UK only for cleaner segmentation
df_uk = df[df['Country'] == 'United Kingdom'].copy()
print(f'\n✅ UK-only dataset: {df_uk.shape[0]:,} rows, {df_uk["Customer ID"].nunique():,} unique customers')

---
## 5. Exploratory Data Analysis (EDA) <a id='5'></a>

In [ ]:
# ── 5.1 Revenue Over Time ─────────────────────────────
df_uk['YearMonth'] = df_uk['InvoiceDate'].dt.to_period('M')
monthly_revenue = df_uk.groupby('YearMonth')['TotalPrice'].sum().reset_index()
monthly_revenue['YearMonth'] = monthly_revenue['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_revenue['YearMonth'], monthly_revenue['TotalPrice'],
        color='#0D7377', linewidth=2.5, marker='o', markersize=4)
ax.fill_between(range(len(monthly_revenue)), monthly_revenue['TotalPrice'],
                alpha=0.15, color='#0D7377')
ax.set_xticks(range(len(monthly_revenue)))
ax.set_xticklabels(monthly_revenue['YearMonth'], rotation=45, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))
ax.set_title('Monthly Revenue — UK Customers', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2 Top 10 Products by Revenue ────────────────────
top_products = (df_uk.groupby('Description')['TotalPrice']
                .sum().sort_values(ascending=False).head(10))

fig, ax = plt.subplots(figsize=(11, 5))
top_products.sort_values().plot(kind='barh', color='#1B2A4A', edgecolor='white', ax=ax)
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue (£)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 Orders by Day of Week & Hour ──────────────────
df_uk['DayOfWeek'] = df_uk['InvoiceDate'].dt.day_name()
df_uk['Hour']      = df_uk['InvoiceDate'].dt.hour

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df_uk.groupby('DayOfWeek')['Invoice'].nunique().reindex(day_order)
hour_counts = df_uk.groupby('Hour')['Invoice'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

day_counts.plot(kind='bar', color='#0D7377', edgecolor='white', ax=axes[0])
axes[0].set_title('Orders by Day of Week', fontsize=13, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of Orders')
axes[0].tick_params(axis='x', rotation=30)

hour_counts.plot(kind='bar', color='#E74C3C', edgecolor='white', ax=axes[1])
axes[1].set_title('Orders by Hour of Day', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Number of Orders')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.4 Revenue Distribution per Customer ─────────────
customer_revenue = df_uk.groupby('Customer ID')['TotalPrice'].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(customer_revenue, bins=80, color='#1B2A4A', edgecolor='white', alpha=0.85)
axes[0].set_title('Revenue Distribution per Customer (Full Range)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Revenue (£)')
axes[0].set_ylabel('Number of Customers')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))

axes[1].hist(customer_revenue[customer_revenue < 5000], bins=60,
             color='#0D7377', edgecolor='white', alpha=0.85)
axes[1].set_title('Revenue Distribution (Capped at £5K — majority)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Revenue (£)')
axes[1].set_ylabel('Number of Customers')

plt.tight_layout()
plt.show()

print(f'\nRevenue Stats per Customer:')
print(customer_revenue.describe().apply(lambda x: f'£{x:,.2f}').to_string())

---
## 6. RFM Feature Engineering <a id='6'></a>

**Why RFM?**  
RFM is a proven, business-grounded framework used across e-commerce, banking, and retail. By computing Recency, Frequency, and Monetary value per customer, we create a compact 3-dimensional representation of customer behaviour that is both interpretable and highly predictive of future value.

In [ ]:
# ── 6.1 Compute RFM metrics ───────────────────────────
# Reference date = 1 day after last transaction (standard practice)
reference_date = df_uk['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Reference Date: {reference_date.date()}')

rfm = df_uk.groupby('Customer ID').agg(
    Recency   = ('InvoiceDate',  lambda x: (reference_date - x.max()).days),
    Frequency = ('Invoice',      'nunique'),
    Monetary  = ('TotalPrice',   'sum')
).reset_index()

print(f'\nRFM Table Shape: {rfm.shape}')
print(f'Unique Customers: {rfm.shape[0]:,}')
rfm.head(10)

In [ ]:
# ── 6.2 RFM Summary Statistics ────────────────────────
print('📊 RFM Summary Statistics')
print('─' * 45)
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2).to_string())

In [ ]:
# ── 6.3 RFM Distributions ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ['#E74C3C', '#0D7377', '#F39C12']
labels = ['Recency (days since last purchase)',
          'Frequency (number of orders)',
          'Monetary (total spend £)']
cols   = ['Recency', 'Frequency', 'Monetary']

for i, (col, color, label) in enumerate(zip(cols, colors, labels)):
    axes[i].hist(rfm[col], bins=50, color=color, edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=13, fontweight='bold')
    axes[i].set_xlabel(label, fontsize=9)
    axes[i].set_ylabel('Number of Customers')
    axes[i].axvline(rfm[col].median(), color='black', linestyle='--',
                    linewidth=1.5, label=f'Median: {rfm[col].median():.0f}')
    axes[i].legend(fontsize=9)

plt.suptitle('RFM Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.4 Handle Skewness — Log Transform ───────────────
# RFM features are heavily right-skewed
# Log transform normalises them for better K-Means performance

rfm['Recency_log']   = np.log1p(rfm['Recency'])
rfm['Frequency_log'] = np.log1p(rfm['Frequency'])
rfm['Monetary_log']  = np.log1p(rfm['Monetary'])

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for i, col in enumerate(['Recency', 'Frequency', 'Monetary']):
    axes[0, i].hist(rfm[col], bins=50, color='#E74C3C', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'{col} — Original', fontweight='bold')
    axes[0, i].set_ylabel('Count')

    axes[1, i].hist(rfm[f'{col}_log'], bins=50, color='#0D7377', edgecolor='white', alpha=0.8)
    axes[1, i].set_title(f'{col} — Log Transformed', fontweight='bold')
    axes[1, i].set_ylabel('Count')

plt.suptitle('Before vs After Log Transformation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.5 Scale Features ───────────────────────────────
rfm_features = rfm[['Recency_log', 'Frequency_log', 'Monetary_log']].copy()

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)
rfm_scaled_df = pd.DataFrame(rfm_scaled,
                              columns=['Recency_scaled', 'Frequency_scaled', 'Monetary_scaled'])

print('✅ Features log-transformed and standardised')
print(f'Mean after scaling:  {rfm_scaled_df.mean().round(4).to_dict()}')
print(f'Std  after scaling:  {rfm_scaled_df.std().round(4).to_dict()}')

---
## 7. K-Means Clustering <a id='7'></a>

In [ ]:
# ── 7.1 Elbow Method ──────────────────────────────────
# Find optimal number of clusters using inertia (WCSS)

inertias = []
K_range  = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(K_range, inertias, marker='o', color='#1B2A4A', linewidth=2.5, markersize=8)
ax.fill_between(K_range, inertias, alpha=0.08, color='#1B2A4A')
ax.set_title('Elbow Method — Optimal Number of Clusters', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia (WCSS)')
ax.set_xticks(list(K_range))
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.2 Silhouette Score ──────────────────────────────
# Validates cluster quality — higher is better (max = 1)

sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(rfm_scaled)
    score  = silhouette_score(rfm_scaled, labels)
    sil_scores.append(score)
    print(f'K={k}  →  Silhouette Score: {score:.4f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(K_range, sil_scores, marker='s', color='#0D7377', linewidth=2.5, markersize=8)
best_k = K_range[sil_scores.index(max(sil_scores))]
ax.axvline(best_k, color='#E74C3C', linestyle='--', linewidth=1.5,
           label=f'Best K = {best_k} (score={max(sil_scores):.4f})')
ax.set_title('Silhouette Score by Number of Clusters', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Silhouette Score')
ax.set_xticks(list(K_range))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print(f'\n✅ Recommended K based on Silhouette: {best_k}')

In [ ]:
# ── 7.3 Final K-Means Model ───────────────────────────
# We use K=4 — balances statistical validity with business interpretability
# 4 segments map cleanly to Champions / Loyal / At Risk / Lost

OPTIMAL_K = 4

kmeans_final = KMeans(n_clusters=OPTIMAL_K, init='k-means++',
                      n_init=10, random_state=42)
rfm['Cluster'] = kmeans_final.fit_predict(rfm_scaled)

print(f'✅ K-Means fitted with K={OPTIMAL_K}')
print(f'\nCluster sizes:')
print(rfm['Cluster'].value_counts().sort_index().to_string())
print(f'\nInertia: {kmeans_final.inertia_:.2f}')
sil = silhouette_score(rfm_scaled, rfm['Cluster'])
print(f'Silhouette Score: {sil:.4f}')

In [ ]:
# ── 7.4 PCA Visualisation of Clusters ─────────────────
# Reduce 3D RFM space to 2D for visual inspection

pca = PCA(n_components=2, random_state=42)
rfm_pca = pca.fit_transform(rfm_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
colors_map = {0:'#E74C3C', 1:'#0D7377', 2:'#F39C12', 3:'#8E44AD'}

for cluster_id in sorted(rfm['Cluster'].unique()):
    mask = rfm['Cluster'] == cluster_id
    ax.scatter(rfm_pca[mask, 0], rfm_pca[mask, 1],
               c=colors_map[cluster_id], label=f'Cluster {cluster_id}',
               alpha=0.5, s=18, edgecolors='none')

ax.set_title('Customer Clusters — PCA Projection (3D RFM → 2D)',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(title='Cluster', fontsize=11)
plt.tight_layout()
plt.show()

print(f'Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%')

---
## 8. Cluster Analysis & Profiling <a id='8'></a>

In [ ]:
# ── 8.1 Cluster Summary Statistics ───────────────────
cluster_summary = rfm.groupby('Cluster').agg(
    Customers  = ('Customer ID', 'count'),
    Recency_Avg= ('Recency',    'mean'),
    Freq_Avg   = ('Frequency',  'mean'),
    Monetary_Avg=('Monetary',   'mean'),
    Monetary_Sum=('Monetary',   'sum')
).round(1)

cluster_summary['% Customers'] = (cluster_summary['Customers'] /
                                   cluster_summary['Customers'].sum() * 100).round(1)
cluster_summary['% Revenue']   = (cluster_summary['Monetary_Sum'] /
                                   cluster_summary['Monetary_Sum'].sum() * 100).round(1)

print('📊 Cluster Summary')
print(cluster_summary.to_string())

In [ ]:
# ── 8.2 Assign Segment Names ──────────────────────────
# Based on the RFM profile of each cluster:
# Low Recency  = bought recently (GOOD)
# High Frequency = buys often    (GOOD)
# High Monetary  = spends more   (GOOD)

# Sort clusters by Monetary value descending to assign names
rank_order = cluster_summary.sort_values('Monetary_Avg', ascending=False).index.tolist()

segment_map = {
    rank_order[0]: 'Champions',
    rank_order[1]: 'Loyal Customers',
    rank_order[2]: 'At Risk',
    rank_order[3]: 'Lost / Inactive',
}

rfm['Segment'] = rfm['Cluster'].map(segment_map)

print('📋 Cluster → Segment Mapping:')
for k, v in segment_map.items():
    row = cluster_summary.loc[k]
    print(f'  Cluster {k} → {v:18s} | Avg Recency: {row["Recency_Avg"]:.0f}d | '
          f'Avg Frequency: {row["Freq_Avg"]:.1f} | Avg Spend: £{row["Monetary_Avg"]:.0f}')

In [ ]:
# ── 8.3 Segment Size & Revenue Share ─────────────────
seg_stats = rfm.groupby('Segment').agg(
    Customers=('Customer ID', 'count'),
    Total_Revenue=('Monetary', 'sum')
).reset_index()
seg_stats['Pct_Customers'] = seg_stats['Customers'] / seg_stats['Customers'].sum() * 100
seg_stats['Pct_Revenue']   = seg_stats['Total_Revenue'] / seg_stats['Total_Revenue'].sum() * 100

seg_colors = {
    'Champions':      '#0D7377',
    'Loyal Customers':'#2980B9',
    'At Risk':        '#F39C12',
    'Lost / Inactive':'#E74C3C',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Customer share
colors_list = [seg_colors[s] for s in seg_stats['Segment']]
axes[0].pie(seg_stats['Pct_Customers'], labels=seg_stats['Segment'],
            autopct='%1.1f%%', colors=colors_list, startangle=140,
            wedgeprops={'edgecolor':'white', 'linewidth':2})
axes[0].set_title('Customer Share by Segment', fontsize=13, fontweight='bold')

# Revenue share
axes[1].pie(seg_stats['Pct_Revenue'], labels=seg_stats['Segment'],
            autopct='%1.1f%%', colors=colors_list, startangle=140,
            wedgeprops={'edgecolor':'white', 'linewidth':2})
axes[1].set_title('Revenue Share by Segment', fontsize=13, fontweight='bold')

plt.suptitle('Customer vs Revenue Distribution by Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.4 RFM Box Plots by Segment ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 6))

seg_order = ['Champions', 'Loyal Customers', 'At Risk', 'Lost / Inactive']
palette   = [seg_colors[s] for s in seg_order]

metrics = [
    ('Recency',   'Recency (days since last purchase)', 'Lower = Better'),
    ('Frequency', 'Frequency (number of orders)',       'Higher = Better'),
    ('Monetary',  'Monetary (total spend £)',           'Higher = Better'),
]

for i, (col, ylabel, note) in enumerate(metrics):
    sns.boxplot(data=rfm, x='Segment', y=col, order=seg_order,
                palette=palette, ax=axes[i], linewidth=1.2)
    axes[i].set_title(f'{col} by Segment\n({note})', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel(ylabel, fontsize=9)
    axes[i].tick_params(axis='x', rotation=20)

plt.suptitle('RFM Distribution by Customer Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.5 Snake Plot — Standardised RFM by Segment ─────
# Industry-standard visualisation for comparing segments across all RFM dimensions

rfm_scaled_df2 = pd.DataFrame(rfm_scaled,
                               columns=['Recency_scaled', 'Frequency_scaled', 'Monetary_scaled'])
rfm_scaled_df2['Segment'] = rfm['Segment'].values

snake = rfm_scaled_df2.groupby('Segment')[['Recency_scaled','Frequency_scaled','Monetary_scaled']].mean()

fig, ax = plt.subplots(figsize=(10, 6))
for seg in seg_order:
    if seg in snake.index:
        ax.plot(['Recency', 'Frequency', 'Monetary'],
                snake.loc[seg].values,
                marker='o', linewidth=2.5, markersize=10,
                color=seg_colors[seg], label=seg)

ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_title('Snake Plot — Standardised RFM by Segment', fontsize=14, fontweight='bold')
ax.set_ylabel('Standardised Score (mean=0)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.6 Final Segment Profile Table ──────────────────
final_profile = rfm.groupby('Segment').agg(
    Customers    = ('Customer ID', 'count'),
    Avg_Recency  = ('Recency',    'mean'),
    Avg_Frequency= ('Frequency',  'mean'),
    Avg_Monetary = ('Monetary',   'mean'),
    Total_Revenue= ('Monetary',   'sum')
).round(1)

final_profile['% Customers'] = (final_profile['Customers'] /
                                  final_profile['Customers'].sum() * 100).round(1)
final_profile['% Revenue']   = (final_profile['Total_Revenue'] /
                                  final_profile['Total_Revenue'].sum() * 100).round(1)

final_profile = final_profile.reindex(seg_order)
print('📊 Final Segment Profiles')
print('─' * 90)
print(final_profile.to_string())

---
## 9. Business Recommendations <a id='9'></a>

In [ ]:
# ── 9.1 Segment Strategy Visualisation ───────────────
fig, ax = plt.subplots(figsize=(13, 7))

segments = ['Champions', 'Loyal Customers', 'At Risk', 'Lost / Inactive']
strategies = [
    'Reward with loyalty perks\nAsk for reviews & referrals\nOffer early access to new products',
    'Upsell higher-value products\nSend personalised recommendations\nOffer membership / subscription',
    'Send win-back email campaigns\nOffer time-limited discount (15-20%)\nIdentify reason for disengagement',
    'Re-engagement campaign (big offer)\nIf no response → stop spending\nAnalyse why they churned',
]
icons   = ['🏆', '💛', '⚠️', '😴']
bg_colors = ['#EBF8F7', '#EBF4FF', '#FEF9E7', '#FDECEA']
bd_colors = ['#0D7377', '#2980B9', '#F39C12', '#E74C3C']

ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Segment Strategy Map', fontsize=16, fontweight='bold', pad=15)

for i, (seg, strat, icon, bgc, bdc) in enumerate(
        zip(segments, strategies, icons, bg_colors, bd_colors)):
    x = (i % 2) * 5.1 + 0.2
    y = 4.1 if i < 2 else 0.2
    rect = plt.Rectangle((x, y), 4.7, 3.6, linewidth=2,
                          edgecolor=bdc, facecolor=bgc, zorder=2)
    ax.add_patch(rect)
    ax.text(x+0.25, y+3.15, icon + '  ' + seg,
            fontsize=13, fontweight='bold', color=bdc, va='top')
    ax.text(x+0.25, y+2.65, strat,
            fontsize=9.5, color='#374151', va='top', linespacing=1.6)

plt.tight_layout()
plt.show()

### 📋 Detailed Recommendations by Segment

#### 🏆 Champions
- **Who:** Bought recently, buy frequently, spend the most
- **Strategy:** Reward and retain — these customers are your highest LTV
- **Actions:** VIP loyalty programme, exclusive early access, ask for product reviews and referrals
- **KPI to track:** Churn rate, average order value

#### 💛 Loyal Customers
- **Who:** Buy regularly, decent spend, somewhat recent
- **Strategy:** Upsell and deepen the relationship
- **Actions:** Personalised product recommendations, subscription offers, premium tier upsell
- **KPI to track:** Upgrade rate, order frequency trend

#### ⚠️ At Risk
- **Who:** Used to buy often but haven't recently — showing signs of churn
- **Strategy:** Win-back campaign before they fully disengage
- **Actions:** "We miss you" email with 15-20% discount, personalised reminder of past purchases
- **KPI to track:** Re-activation rate within 30 days

#### 😴 Lost / Inactive
- **Who:** Haven't bought in a long time, low frequency, low spend
- **Strategy:** One final re-engagement push — if no response, stop spending budget on them
- **Actions:** Large discount offer (30%+) as last resort; survey to understand why they left
- **KPI to track:** Re-activation rate; if <5%, remove from active marketing

---
## 10. Conclusion <a id='10'></a>

### 📊 Project Summary

| Step | Action | Output |
|---|---|---|
| Data Cleaning | Removed cancellations, nulls, invalid entries | ~300K clean UK transactions |
| EDA | Revenue trends, top products, order patterns | Business context established |
| RFM Engineering | Computed R, F, M per customer | Customer-level dataset |
| Log Transform | Reduced skewness in all 3 features | Improved clustering quality |
| K-Means | Elbow + Silhouette to pick K=4 | 4 distinct customer segments |
| PCA | 2D visualisation of clusters | Confirmed separation |
| Profiling | Snake plot + box plots per segment | Rich segment characterisation |
| Recommendations | Per-segment marketing strategies | Actionable business output |

### 🔑 Key Findings
1. **Champions** are a small % of customers but drive a disproportionate share of revenue — protecting them is the highest priority
2. **At Risk** customers represent significant recoverable revenue — a targeted win-back campaign has high ROI
3. **RFM + K-Means** together produce interpretable, business-ready segments without needing labelled data
4. **Log transformation** was essential — raw RFM features were too skewed for K-Means to work well
5. **PCA visualisation** confirmed that the 4 clusters are genuinely well-separated in 3D RFM space

### ✅ Project Deliverables
- **Notebook:** `customer_segmentation.ipynb` — Full pipeline from raw data to segments
- **PDF Report:** `Customer_Segmentation_Report.pdf` — Business-ready summary
- **PPTX:** `Customer_Segmentation_Summary.pptx` — Stakeholder presentation
- **README:** `README.md` — GitHub documentation

---
*End of Notebook*